# Email Agent

Creating an agent that authenticates the user, reads emails from an inbox and drafts a response for HITL review.

#### Design

- Utilize agent state to store the user authentication, create and manage the email state
- Develop a tool that reads the email (use @wrap_model_call to base access on the user type)
- A tool that drafts an email and is interrupted for human review and approval

In [11]:
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware, ModelRequest, ModelResponse, wrap_model_call
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command, Callable
from langchain.messages import HumanMessage, ToolMessage

from dataclasses import dataclass

from dotenv import load_dotenv

from typing import Dict, Any

In [12]:
load_dotenv()

True

## Setting Up Context and Tools

Next steps:
- Create tools to write passed state objects to the agent graph

In [3]:
class EmailUser(AgentState):
    user_auth: str
    inbox: Dict[str, Any]
    outbox: Dict[str, Any]

In [ ]:
@tool
def create_user_inbox(runtime: ToolRuntime,
                      user_auth: str,
                      inbox: Dict[str, Any],
                      outbox: Dict[str, Any]) -> str:
    """Updates the user's inbox, outbox and authorization status.
    Use this tool to update the state when the information is given."""

    try:
        return Command(update={
            "inbox": inbox,
            "outbox": outbox,
            "user_auth": user_auth,
            "messages": [ToolMessage(
                content="Inbox, outbox and user_auth created successfully",
                tool_call_id=runtime.tool_call_id,
        )],
    })
    except Exception as e:
        return f"Could not create inbox, error: {e}"

@tool
def read_inbox(runtime: ToolRuntime) -> str:
    """Reads emails from the user's inbox."""

    try:
        return runtime.state["inbox"]
    except Exception as e:
        return f"Could not read inbox, error: {e}"


@wrap_model_call
def read_inbox_permission(runtime: ToolRuntime,
                          request: ModelRequest,
                          handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Gives permission to the agent to read the user's inbox if they are authorized"""

    if runtime.state["user_auth"] == "authorized":
        pass
    else:
        tools = None
        request = request.override(tools=tools)

    return handler(request)

@tool
def send_email(runtime: ToolRuntime, body: str) -> Command:
    """Adds a reply to the outbox at the next position."""
    outbox = dict(runtime.state.get("outbox", {}))
    next_position = max((int(k) for k in outbox), default=-1) + 1
    outbox[str(next_position)] = body   # keep keys as strings, consistent with how they're stored
    return Command(update={
        "outbox": outbox,
        "messages": [ToolMessage(content="Email sent successfully", tool_call_id=runtime.tool_call_id)],
    })

## Building the Agent

In [5]:
email_agent = create_agent(
    model="claude-haiku-4-5",
    tools=[create_user_inbox, read_inbox, send_email],
    state_schema=EmailUser,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware[AgentState, None](
            interrupt_on={
                "read_inbox": False,
                "send_email": True
            },
            description_prefix="Tool execution requires approval"
        )
    ],
    system_prompt="""You are a helpful email assistant. You will read my inbox
    and automatically draft responses one at a time, starting with the first emial. 
    Do not ask me for confirmation of the written content, an approval step already
    exists."""
)

In [7]:
user_auth = "authorized"
inbox = {0: {"Hey, can we get some time to talk about the agent project? I am free at 2:00 p.m. CST today."},
         1: {"Please see attached for an update on project X. We are on track to deliver ahead of schedule and below budget!"}}
outbox = {0: {"This is the first message I have sent!"}}

In [7]:
config={"configurable": {"thread_id": "1"}}

response = email_agent.invoke(
    {"messages": [HumanMessage(content=f"{user_auth}, {inbox}, {outbox}")]},
    config=config
)

response

{'messages': [HumanMessage(content="authorized, {0: {'Hey, can we get some time to talk about the agent project? I am free at 2:00 p.m. CST today.'}, 1: {'Please see attached for an update on project X. We are on track to deliver ahead of schedule and below budget!'}}, {0: {'This is the first message I have sent!'}}", additional_kwargs={}, response_metadata={}, id='265fa3d4-af32-42de-b406-8da14b53215d'),
  AIMessage(content=[{'text': "I'll read your inbox and start drafting responses to your emails.", 'type': 'text'}, {'id': 'toolu_01VYRBHP4GavLYeVjVTAC8oG', 'caller': {'type': 'direct'}, 'input': {}, 'name': 'read_inbox', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_011CevnwEHZWidKoK3qS62Y2', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_

In [8]:
response = email_agent.invoke(
    Command[tuple[()]](
        resume={"decisions": [{"type": "approve"}]}
    ),
    config=config
)

response

{'messages': [HumanMessage(content="authorized, {0: {'Hey, can we get some time to talk about the agent project? I am free at 2:00 p.m. CST today.'}, 1: {'Please see attached for an update on project X. We are on track to deliver ahead of schedule and below budget!'}}, {0: {'This is the first message I have sent!'}}", additional_kwargs={}, response_metadata={}, id='265fa3d4-af32-42de-b406-8da14b53215d'),
  AIMessage(content=[{'text': "I'll read your inbox and start drafting responses to your emails.", 'type': 'text'}, {'id': 'toolu_01VYRBHP4GavLYeVjVTAC8oG', 'caller': {'type': 'direct'}, 'input': {}, 'name': 'read_inbox', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_011CevnwEHZWidKoK3qS62Y2', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_

# Redesign

- Make the authentication step more robust by setting the default auth status to no
- Revamp the EmailContext class to contain the email address and password of the inbox user
- Use the Agent state to track the authentication status of the user (authenticated if the correct email and passwords are passed as context)
- Dedicated tools that do the following:
  - Reading emails
  - Sending emails
  - Authenticating users --> dynamically updates the agent's state
- Dynamic tool call that provides the appropriate tools based on the user's authentication status
- (Optional) dynamic prompt to give teh agent specific instructions based on the user's authentication status

## Setting Up Agent State and Context

In [21]:
@dataclass
class EmailContext:
    username: str
    password: str

class UserEmail(AgentState):
    authentication: bool = False
    user_inbox: dict
    user_outbox: dict

In [22]:
@tool
def read_email(runtime: ToolRuntime) -> Dict[str, Any] | str:
    """Reads a user's email inbox and outbox if they are successfully authenticated"""

    if runtime.state["authentication"]:
        return runtime.state["user_inbox"], runtime.state["user_outbox"]
    else:
        return "User is not authorized to access the inbox and outbox"


@tool
def send_email(runtime: ToolRuntime, body: str, email_position) -> Command:
    """Sends a given email and writes the message to the outbox."""

    if runtime.state["authentication"]:
        outbox = dict(runtime.state.get("user_outbox", {}))
        outbox[str(email_position)] = body   # keep keys as strings, consistent with how they're stored
        return Command(update={
            "user_outbox": outbox,
            "messages": [ToolMessage(content="Email sent successfully", tool_call_id=runtime.tool_call_id)],
        })
    else:
        return Command({
            "messages": [ToolMessage(content="User is not authorized to send emails", tool_call_id=runtime.tool_call_id)]
        })
        

@tool
def authenticate_user(runtime: ToolRuntime) -> Command:
    """Authenticates the user using credentials already provided via context.
    Call this tool directly with no arguments — do not ask the user for a
    username or password, they are supplied automatically."""

    auth_un, auth_pw = "yourein@right.com", "password123"

    if runtime.context.username == auth_un and runtime.context.password == auth_pw:
        return Command(update={
            "authentication": True,
            "messages": [ToolMessage(
                "Authenticated email user",
                tool_call_id=runtime.tool_call_id
            )]
        })
    else:
        return Command(update={
            "authentication": False,
            "messages": [ToolMessage(
                "Did not authenticate email user",
                tool_call_id=runtime.tool_call_id
            )]
        })

@wrap_model_call
def update_permissions(request: ModelRequest,
                       handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Updates the agent's tools to read and write emails if the user is authenticated"""

    if request.state.get("authentication"):
        tools = [read_email, send_email]
    else:
        tools = [authenticate_user]

    request = request.override(tools=tools)
    return handler(request)

In [23]:
updated_agent = create_agent(
    model="claude-haiku-4-5",
    tools=[authenticate_user, read_email, send_email],
    state_schema=UserEmail,
    context_schema=EmailContext,
    checkpointer=InMemorySaver(),
    middleware=[
        update_permissions,
        HumanInTheLoopMiddleware[AgentState, None](
            interrupt_on={
                "read_email": False,
                "authenticate_user": False,
                "send_email": True
            },
            description_prefix="Tool execution requires approval"
        )
    ],
    system_prompt="""You are a helpful email assistant. The user's credentials are
already available to you — call authenticate_user immediately, with no
arguments, before doing anything else. Do not ask the user for a username
or password. Once authenticated, read my inbox and automatically draft
responses one at a time, starting with the first email. Do not ask me for
confirmation of the written content, an approval step already exists."""
)

In [24]:
config2 = {"configurable": {"thread_id": "2"}}

updated_response = updated_agent.invoke(
    {"messages": [HumanMessage(content="Can you read my inbox and send a reply to the first email?")],
     "user_inbox": inbox,
     "user_outbox": outbox},
    config=config2,
    context=EmailContext(username="yourein@right.com", password="password123")
)

/Users/samueljoseph/Documents/Programming/agent-lab/.venv/lib/python3.13/site-packages/pydantic/functional_validators.py:835: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=EmailContext(username='yo... password='password123'), input_type=EmailContext])
  function=lambda v, h: h(v), schema=original_schema
/Users/samueljoseph/Documents/Programming/agent-lab/.venv/lib/python3.13/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=EmailContext(username='yo... password='password123'), input_type=EmailContext])
  return self.__pydantic_serializer__.to_python(
/Users/samueljoseph/Documents/Programming/agent-lab/.venv/lib/python3.13/site-packages/pydantic/functional_validators.py:835: UserWarning: Pydantic seria

In [25]:
updated_response = updated_agent.invoke(
    Command[tuple[()]](
        resume={"decisions": [{"type": "approve"}]}
    ),
    config=config2
)

updated_response

{'messages': [HumanMessage(content='Can you read my inbox and send a reply to the first email?', additional_kwargs={}, response_metadata={}, id='fe40909d-d8a5-437b-bb90-29faf95234b0'),
  AIMessage(content=[{'text': "I'll start by authenticating you, then read your inbox and draft a reply to the first email.", 'type': 'text'}, {'id': 'toolu_01LUAXXyuLX7Ts8dYHp4iznF', 'caller': {'type': 'direct'}, 'input': {}, 'name': 'authenticate_user', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_011Ceydrwtvkhx9buGmWLZkb', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 681, 'output_tokens': 58, 'output_tokens_details': None, 'server_tool_use': None, 'service_tier': 'standard'}, 'model_nam